In [ ]:
# Find this checkout from a notebook or a kernel launched at the workspace root.
from pathlib import Path
import os, sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents, _here / 'cic17-cic18-transfer-study']
if os.environ.get('NIDS_STUDY_ROOT'):
    _candidates.insert(0, Path(os.environ['NIDS_STUDY_ROOT']).expanduser().resolve())
REPO_ROOT = next((p for p in _candidates if (p / 'src/nids_transfer').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this checkout or set NIDS_STUDY_ROOT.')
_src = str(REPO_ROOT / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)
from nids_transfer.paths import study_paths
DATA_DIR, ARTIFACT_DIR = study_paths(REPO_ROOT)
print('Data:', DATA_DIR)
print('Artifacts:', ARTIFACT_DIR)


# Binary detection baseline: 2017 training → held-out 2017 and 2018

**Question:** how much does attack detection change when the same model moves from its source dataset to the other year?

1. Freeze an approximately 80/20 **row-level split of cleaned 2017**, stratified by original label to preserve rare attacks as far as possible.
2. Fit all preprocessing on the 80% training partition only. Leave 2018 and held-out 2017 out of fitting and model selection.
3. Fit logistic regression and a resource-bounded random forest using the full training pool.
4. Predict on **every cleaned evaluation row**, globally and by original attack category. Save predictions and stable row IDs for the next notebook.

This is an initial random-flow baseline, not a session/time/capture-independent generalization experiment. Closely related or duplicate flows can cross the row split. We audit transformed feature-hash matches and show evaluation both with and without such matches; a grouped/time split remains a later stronger check. No category-mixing or clustering analysis is used here.

Run in `IMT_data_analyser` with NumPy, pandas, DuckDB, PyArrow, scikit-learn, joblib, and matplotlib. Helper code: `src/nids_transfer/binary_transfer_utils.py`; baseline constructors were extracted from the original `ml_models.py` into `src/nids_transfer/models.py`. Source Parquet files are read-only. Derived matrices/models are stored under `binary_baseline_results/`. Disk-backed matrices need about 1 GB; fitting estimators still needs working RAM.

In [ ]:
from pathlib import Path
import json, platform, hashlib
import numpy as np
import pandas as pd
import sklearn, duckdb, joblib
from IPython.display import display
from nids_transfer.binary_transfer_utils import BinaryStudy, write_json

ROOT = REPO_ROOT
OUT = ARTIFACT_DIR / 'binary_baseline_results'
PATHS = {year: DATA_DIR / f'compatible/CIC{year}_shared.parquet' for year in ['2017','2018']}
SEED = 42
study = BinaryStudy(PATHS, OUT, seed=SEED)


## Freeze the evaluation populations

The split is stratified by raw attack label, but the classifier target is **0 = BENIGN, 1 = every other label**, including attempted variants. Original labels remain available for reporting. No attack-family equivalence is assumed across years.

Use `year:Parquet source-row-number` as the stable key; flow IDs and the existing `id` field need not be unique. Exclude rows with non-finite numeric measurements (the same finite-row cleaning as the earlier comparison). Counts of exclusions are recorded. Labels must be present.

Artifacts are checkpointed: rerunning complete stages loads saved outputs. Inputs/seed/schema are checked. Use a new output directory for different experiments or preprocessing/model settings; do not mix old artifacts with new settings.

In [ ]:
split_counts = study.freeze_splits()
display(pd.Series(split_counts, name='rows'))
inventory = []
for split in ['train_2017','test_2017','test_2018']:
    rows = pd.read_parquet(OUT / f'{split}_rows.parquet')
    table = rows.groupby('label').size().rename(split)
    inventory.append(table)
label_counts = pd.concat(inventory,axis=1).fillna(0).astype(int)
display(label_counts)
label_counts.to_csv(OUT / 'split_label_counts.csv')
del rows


## Train-only feature preparation, in batches

Exclude IPs, IDs, timestamps, labels/capture metadata, ports, and ICMP categorical codes. Include quantitative flow measurements **and genuine flag counts**. For numeric columns:

- Convert negative values (including negative sentinels) to missing for this nonnegative-flow study; audit them.
- Apply `log1p` to observed nonnegative values.
- Fit `StandardScaler` statistics column by column on training observations only, skipping all-missing batches.
- Mean-impute in log space, equivalent to a zero standardized value. Remove training-constant/all-missing columns.

Protocol uses a full one-hot vocabulary learned on training rows only. Unseen protocols produce an all-zero block and are explicitly flagged in evaluation metadata. There is no PCA in the classifier features and no year/label input.

Store C-order float32 `.npy` matrices on disk. Batch preprocessing and prediction limit memory use; **batching does not make the classifier fit incremental**. The saved feature order and preprocessing object make transformations reproducible.

In [ ]:
matrix_summary = study.prepare_matrices()
display(pd.DataFrame(matrix_summary).T)
preprocessor = joblib.load(OUT / 'preprocessor.joblib')
display(pd.read_csv(OUT / 'preprocessing_audit.csv'))
print('Classifier feature count:', len(preprocessor['feature_names']))
print('Training-only protocol vocabulary:', preprocessor['protocols'])
print('Disk-backed matrices:', [(p.name, round(p.stat().st_size / 2**20,1)) for p in OUT.glob('*_X.npy')])


## Models and compute choices

Use `build_cleanlab_model` from `nids_transfer.models` (the original baseline constructors), with documented overrides:

- **Logistic regression:** extract the factory's logistic estimator, avoiding its redundant second scaler; use SAGA (`max_iter=150`, `tol=0.001`, fixed seed) to retain float32 support. Fit all training rows together. Inspect warnings/iterations below rather than assuming convergence.
- **Random forest:** 100 trees, at most depth 20 / 4,096 leaves per tree, minimum leaf size 5, two workers. Each tree draws 200,000 rows with replacement (or the whole pool size if smaller) **from the full 2017 training pool**. This controls memory/runtime; it is not a forest trained on a single small fixed subset. The training pool is full-size, although an individual tree does not see every row.

No class resampling, target-year adaptation, hyperparameter search, or threshold tuning is performed. Both models use a predeclared **0.5 attack-score threshold**. These are resource-bounded baselines, not claims of optimal performance. Neural models are deferred.

Run the following fit cells separately if needed. Completed models are cached; prediction is separately checkpointed.

In [ ]:
logreg_fit = study.fit_model('logreg')
display(pd.Series({k:v for k,v in logreg_fit.items() if k != 'parameters'}))
print(logreg_fit['parameters'])


In [ ]:
rf_fit = study.fit_model('rf')
display(pd.Series({k:v for k,v in rf_fit.items() if k != 'parameters'}))
print(rf_fit['parameters'])


## Frozen full-evaluation predictions

Batch prediction covers all held-out 2017 rows and all cleaned 2018 rows. The threshold and fitted preprocessing are unchanged between datasets.

Global metrics include confusion counts, attack recall, benign false-positive rate (FPR), precision, F1, balanced accuracy, ROC-AUC and average precision. Also report a sensitivity view excluding evaluation inputs whose **64-bit hash of the prepared float32 features** occurs in training. A hash match is a duplicate/identical-input candidate, not a cryptographic proof of identical raw flows; it can also reflect information removed by preprocessing.

Per attack, report **detection recall**, not multiclass accuracy. For BENIGN, report FPR. Mark raw labels absent from training. Wilson 95% intervals are pointwise binomial intervals; correlated flows can make these too narrow. Single-class subsets have undefined ROC-AUC/balanced accuracy, shown as missing.

PR-based metrics and precision depend on traffic prevalence. Always keep counts next to comparisons; category difficulty and category prevalence are different effects.

In [ ]:
reports = {}
for model in ['logreg','rf']:
    overall, per_label = study.predict(model)
    reports[model] = (overall, per_label)
overall_results = pd.concat([v[0] for v in reports.values()],ignore_index=True)
per_label_results = pd.concat([v[1] for v in reports.values()],ignore_index=True)
display(overall_results)
display(per_label_results)
overall_results.to_csv(OUT / 'baseline_overall.csv',index=False)
per_label_results.to_csv(OUT / 'baseline_per_label.csv',index=False)


In [ ]:
import matplotlib.pyplot as plt
for model in ['logreg','rf']:
    table = per_label_results.query('model == @model')
    attacks = table.query("label != 'BENIGN'")
    labels = sorted(attacks.label.unique())
    fig, ax = plt.subplots(figsize=(12,max(5,len(labels)*.28)),constrained_layout=True)
    for offset,(split,color) in zip([-.12,.12],[('test_2017','tab:blue'),('test_2018','tab:orange')]):
        g = attacks.query('split == @split').set_index('label').reindex(labels)
        ax.errorbar(g.value,np.arange(len(labels))+offset,
                    xerr=np.vstack([g.value-g.ci_low,g.ci_high-g.value]),
                    fmt='o',ms=3,capsize=2,color=color,label=split)
    ax.set_yticks(np.arange(len(labels)),labels);ax.set_xlim(-.02,1.02)
    ax.set_xlabel('Attack detection recall (pointwise Wilson 95% interval)')
    ax.set_title(model+' — unseen raw labels may be present in 2018');ax.legend();ax.grid(alpha=.2)
    plt.show()


In [ ]:
registry = {
    'seed':SEED,'models':['logreg','rf'],'threshold':.5,'split_policy':'2017 raw-label-stratified random rows, 80/20',
    'counts':split_counts,'features':preprocessor['feature_names'],
    'versions':{'python':platform.python_version(),'sklearn':sklearn.__version__,'numpy':np.__version__,
                'pandas':pd.__version__,'duckdb':duckdb.__version__},
    'helper_sha256':hashlib.sha256((REPO_ROOT/'src/nids_transfer/binary_transfer_utils.py').read_bytes()).hexdigest(),
    'model_factory_sha256':hashlib.sha256((REPO_ROOT/'src/nids_transfer/models.py').read_bytes()).hexdigest(),
}
# Keep the original provenance when reusing an already-fitted local run.
_registry_path = OUT/'baseline_complete.json'
if _registry_path.exists():
    _existing = json.loads(_registry_path.read_text())
    for _key in ['seed','models','threshold','split_policy','counts','features']:
        if _existing.get(_key) != registry.get(_key):
            raise ValueError(f'Cached baseline registry differs at {_key}; choose a new artifact directory.')
    registry = _existing
    print('Retaining the original fitted-run provenance and coverage cache identity.')
else:
    write_json(_registry_path,registry)
print('Ready for coverage_and_transfer_analysis.ipynb:',OUT)


## Reading this recorded run

The following summary is generated from the saved tables and updates when those tables are regenerated.


In [ ]:
# Read the frozen full-evaluation results, without fitting again.
recorded = pd.read_csv(OUT/'baseline_overall.csv')
for model in ['logreg','rf']:
    g=recorded.query('model == @model and scope == "all"').set_index('split')
    print(f"{model}: attack recall {g.loc['test_2017','attack_recall']:.1%} (held-out 2017) -> {g.loc['test_2018','attack_recall']:.1%} (2018)")
    print(f"    benign FPR {g.loc['test_2017','benign_fpr']:.2%} -> {g.loc['test_2018','benign_fpr']:.2%}")
print('This establishes a transfer gap; the coverage notebook investigates an association, not a causal explanation.')


## What this baseline can and cannot say

Compare held-out 2017 with 2018 at the same decision threshold. Inspect recall by attack and benign FPR before drawing conclusions from overall accuracy. The next notebook asks whether errors concentrate where the training population has weak geometric support.

This random-flow baseline does not eliminate session/capture dependence, establish causality, or show that similarly named attack labels are equivalent. Preprocessing discards some information and treats negatives as missing. Check the audit and hash-match sensitivity results. Dataset revisions/labels can differ even within an apparently overlapping region.

Do not select the best model/settings using 2018 scores and then call those scores an untouched final test. Any later tuning should remain inside 2017 training, with a new clearly tracked experiment.